In [1]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import random
import os
import optuna
from optuna.samplers import TPESampler
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler, LabelEncoder, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import StratifiedKFold
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import f1_score

# ============================================================================
# 1. CONFIGURATION & SEEDING
# ============================================================================
def seed_everything(seed=42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

SEED = 42
seed_everything(SEED)

# Check for GPU
if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
    print(f"✅ GPU Detected: {torch.cuda.get_device_name(0)}")
else:
    DEVICE = torch.device("cpu")
    print("⚠️ GPU NOT DETECTED. Running on CPU.")

# Fixed Settings for "Logistic Regression"
BATCH_SIZE = 64
FOLDS = 5
N_TRIALS = 50 # Optuna trials (Logistic Regression is fast to tune)

# ============================================================================
# 2. DATA LOADING & ENGINEERING
# ============================================================================
try:
    train_df = pd.read_csv('train.csv')
    test_df = pd.read_csv('test.csv')
    print("✅ Loaded Data.")
except FileNotFoundError:
    raise FileNotFoundError("❌ Upload train.csv and test.csv!")

# Feature Engineering (Reusing your best logic)
def create_advanced_features(df):
    df = df.copy()
    activity_cols = ['hobby_engagement_level', 'physical_activity_index', 
                     'creative_expression_index', 'altruism_score']
    df['total_activity'] = df[activity_cols].sum(axis=1)
    df['support_guidance_combo'] = df['support_environment_score'] * (df['external_guidance_usage'] + 1)
    df['focus_efficiency'] = df['focus_intensity'] / (df['consistency_score'] + 1)
    df['consistency_gap'] = 30 - df['consistency_score']
    df['focus_sq'] = df['focus_intensity'] ** 2
    df['focus_X_consistency'] = df['focus_intensity'] * df['consistency_score']
    df['low_focus_high_consist'] = ((df['focus_intensity'] < 5) & (df['consistency_score'] > 24)).astype(int)
    return df

train_df = create_advanced_features(train_df)
test_df = create_advanced_features(test_df)

X = train_df.drop(['participant_id', 'personality_cluster'], axis=1)
y = train_df['personality_cluster']
test_ids = test_df['participant_id']
X_test = test_df.drop(['participant_id'], axis=1)

# Encode Target
le = LabelEncoder()
y_encoded = le.fit_transform(y)
num_classes_target = len(le.classes_)

# ============================================================================
# 3. PREPROCESSING (Standard Scaling + One-Hot)
# ============================================================================
# Logistic Regression requires One-Hot Encoding for categoricals, not embeddings.
cat_cols = [
    'identity_code', 'cultural_background', 'age_group', 
    'upbringing_influence', 'support_environment_score', 
    'hobby_engagement_level', 'physical_activity_index',
    'creative_expression_index', 'altruism_score',
    'low_focus_high_consist'
]
num_cols = [c for c in X.columns if c not in cat_cols]

# Pipeline
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), num_cols),
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), cat_cols)
    ]
)

X_processed = preprocessor.fit_transform(X)
X_test_processed = preprocessor.transform(X_test)
input_dim = X_processed.shape[1]

# Class Weights (Critical for Logistic Regression on imbalanced data)
class_weights = compute_class_weight('balanced', classes=np.unique(y_encoded), y=y_encoded)
class_weights_tensor = torch.tensor(class_weights, dtype=torch.float).to(DEVICE)

# ============================================================================
# 4. MODEL & DATASET DEFINITION
# ============================================================================
class LogisticRegressionModel(nn.Module):
    def __init__(self, input_dim, num_classes):
        super(LogisticRegressionModel, self).__init__()
        # A single Linear layer is mathematically equivalent to Logistic Regression
        self.linear = nn.Linear(input_dim, num_classes)
        
    def forward(self, x):
        return self.linear(x)

class TabularDataset(Dataset):
    def __init__(self, X_data, y_data=None):
        self.X = torch.tensor(X_data, dtype=torch.float)
        self.y = torch.tensor(y_data, dtype=torch.long) if y_data is not None else None
    def __len__(self): return len(self.X)
    def __getitem__(self, idx):
        if self.y is not None: return self.X[idx], self.y[idx]
        return self.X[idx]

# ============================================================================
# 5. OPTUNA TUNING
# ============================================================================
print(f"--- Starting Logistic Regression Optuna Tuning ({N_TRIALS} Trials) ---")

def objective(trial):
    # Hyperparameters for Logistic Regression (Regularization & Optimization)
    lr = trial.suggest_float('lr', 1e-4, 1e-1, log=True)
    weight_decay = trial.suggest_float('weight_decay', 1e-6, 1e-1, log=True) # L2 Regularization
    epochs = trial.suggest_int('epochs', 20, 100)
    
    # 3-Fold CV for speed
    skf_tune = StratifiedKFold(n_splits=3, shuffle=True, random_state=SEED)
    scores = []
    
    for train_idx, val_idx in skf_tune.split(X_processed, y_encoded):
        train_ds = TabularDataset(X_processed[train_idx], y_encoded[train_idx])
        val_ds = TabularDataset(X_processed[val_idx], y_encoded[val_idx])
        
        train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
        val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE*2, shuffle=False, num_workers=0)
        
        model = LogisticRegressionModel(input_dim, num_classes_target).to(DEVICE)
        criterion = nn.CrossEntropyLoss(weight=class_weights_tensor)
        optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
        
        for epoch in range(epochs):
            model.train()
            for x_batch, y_batch in train_loader:
                x_batch, y_batch = x_batch.to(DEVICE), y_batch.to(DEVICE)
                optimizer.zero_grad()
                outputs = model(x_batch)
                loss = criterion(outputs, y_batch)
                loss.backward()
                optimizer.step()
        
        # Validate
        model.eval()
        preds, labels = [], []
        with torch.no_grad():
            for x_batch, y_batch in val_loader:
                x_batch = x_batch.to(DEVICE)
                outputs = model(x_batch)
                p = torch.argmax(outputs, dim=1)
                preds.extend(p.cpu().numpy())
                labels.extend(y_batch.numpy())
        
        scores.append(f1_score(labels, preds, average='macro'))
        
    return np.mean(scores)

sampler = TPESampler(seed=SEED)
study = optuna.create_study(direction="maximize", sampler=sampler)
study.optimize(objective, n_trials=N_TRIALS)

print("\n✅ BEST LR PARAMS:")
print(study.best_params)
best_p = study.best_params

# ============================================================================
# 6. FINAL TRAINING & SUBMISSION
# ============================================================================
print("\n--- Retraining Final Logistic Regression Model (5 Folds) ---")

skf_final = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
test_probs_sum = np.zeros((len(X_test), num_classes_target))
final_f1_scores = []

for fold, (train_idx, val_idx) in enumerate(skf_final.split(X_processed, y_encoded)):
    train_ds = TabularDataset(X_processed[train_idx], y_encoded[train_idx])
    val_ds = TabularDataset(X_processed[val_idx], y_encoded[val_idx])
    
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
    val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE*2, shuffle=False, num_workers=0)
    
    model = LogisticRegressionModel(input_dim, num_classes_target).to(DEVICE)
    criterion = nn.CrossEntropyLoss(weight=class_weights_tensor)
    optimizer = optim.AdamW(model.parameters(), lr=best_p['lr'], weight_decay=best_p['weight_decay'])
    
    # Train for best epochs found
    for epoch in range(best_p['epochs']):
        model.train()
        for x_batch, y_batch in train_loader:
            x_batch, y_batch = x_batch.to(DEVICE), y_batch.to(DEVICE)
            optimizer.zero_grad()
            outputs = model(x_batch)
            loss = criterion(outputs, y_batch)
            loss.backward()
            optimizer.step()
            
    # Validate
    model.eval()
    preds, labels = [], []
    with torch.no_grad():
        for x_batch, y_batch in val_loader:
            x_batch = x_batch.to(DEVICE)
            outputs = model(x_batch)
            p = torch.argmax(outputs, dim=1)
            preds.extend(p.cpu().numpy())
            labels.extend(y_batch.numpy())
            
    fold_score = f1_score(labels, preds, average='macro')
    print(f"Fold {fold+1} | F1 Score: {fold_score:.4f}")
    final_f1_scores.append(fold_score)
    
    # Predict on Test
    test_ds = TabularDataset(X_test_processed)
    test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE*2, shuffle=False, num_workers=0)
    
    fold_probs = []
    with torch.no_grad():
        for x_batch in test_loader:
            x_batch = x_batch.to(DEVICE)
            outputs = model(x_batch)
            probs = torch.softmax(outputs, dim=1)
            fold_probs.append(probs.cpu().numpy())
    test_probs_sum += np.concatenate(fold_probs)

# ============================================================================
# 7. SAVING RESULTS
# ============================================================================
print(f"\n🏆 Average Logistic Regression F1: {np.mean(final_f1_scores):.4f}")

avg_test_probs = test_probs_sum / FOLDS

# Save Probs
prob_df = pd.DataFrame(avg_test_probs, columns=[f'prob_{i}' for i in range(num_classes_target)])
prob_df['participant_id'] = test_ids.values
prob_df.to_csv('logreg_optuna_probs.csv', index=False)
print("✅ Saved 'logreg_optuna_probs.csv'")

# Save Submission
final_indices = np.argmax(avg_test_probs, axis=1)
final_labels = le.inverse_transform(final_indices)
submission_df = pd.DataFrame({
    'participant_id': test_ids,
    'personality_cluster': final_labels
})
submission_df.to_csv('submission_logreg.csv', index=False)
print("✅ Saved 'submission_logreg.csv'")

C:\Users\hrush\AppData\Roaming\Python\Python311\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[I 2025-11-28 22:45:43,281] A new study created in memory with name: no-name-938c18ab-cb14-49b1-822c-d4a55a8d726d


✅ GPU Detected: NVIDIA GeForce RTX 3050 Laptop GPU
✅ Loaded Data.
--- Starting Logistic Regression Optuna Tuning (50 Trials) ---


[I 2025-11-28 22:45:55,413] Trial 0 finished with value: 0.4936336582105387 and parameters: {'lr': 0.0013292918943162175, 'weight_decay': 0.05669849511478854, 'epochs': 79}. Best is trial 0 with value: 0.4936336582105387.
[I 2025-11-28 22:45:58,827] Trial 1 finished with value: 0.49281491825232066 and parameters: {'lr': 0.006251373574521752, 'weight_decay': 6.026889128682509e-06, 'epochs': 32}. Best is trial 0 with value: 0.4936336582105387.
[I 2025-11-28 22:46:05,275] Trial 2 finished with value: 0.4247538712048217 and parameters: {'lr': 0.00014936568554617635, 'weight_decay': 0.021423021757741054, 'epochs': 68}. Best is trial 0 with value: 0.4936336582105387.
[I 2025-11-28 22:46:15,851] Trial 3 finished with value: 0.4928491488892464 and parameters: {'lr': 0.013311216080736894, 'weight_decay': 1.2674255898937221e-06, 'epochs': 98}. Best is trial 0 with value: 0.4936336582105387.
[I 2025-11-28 22:46:21,917] Trial 4 finished with value: 0.4745645122612154 and parameters: {'lr': 0.03142


✅ BEST LR PARAMS:
{'lr': 0.0019796084482970188, 'weight_decay': 3.7220857884211096e-06, 'epochs': 100}

--- Retraining Final Logistic Regression Model (5 Folds) ---
Fold 1 | F1 Score: 0.5407
Fold 2 | F1 Score: 0.5020
Fold 3 | F1 Score: 0.5260
Fold 4 | F1 Score: 0.4936
Fold 5 | F1 Score: 0.4758

🏆 Average Logistic Regression F1: 0.5076
✅ Saved 'logreg_optuna_probs.csv'
✅ Saved 'submission_logreg.csv'
